In [ ]:
!pip -q install -U "git+https://github.com/huggingface/diffusers" \
    "transformers>=4.44" accelerate safetensors sentencepiece protobuf \
    bitsandbytes imageio imageio-ffmpeg av peft hf_transfer > /dev/null

import os, gc, time, base64, textwrap, shutil
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_HOME"] = "/content/hf"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from IPython.display import HTML, display

CFG = dict(
    QUANTIZE_TE   = True,
    HEIGHT        = 480,
    WIDTH         = 704,
    NUM_FRAMES    = 97,
    FPS           = 24,
    SEED          = 42,
    RUN_T2V       = True,
    RUN_I2V       = False,
    RUN_TWO_STAGE = False,
)
IMAGE_URL = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/penguin.png"

CANDIDATES = [
    ("Lightricks/LTX-Video-0.9.6-distilled", "distilled"),
    ("Lightricks/LTX-Video-0.9.5",           "dev"),
    ("Lightricks/LTX-Video",                 "dev"),
]
UPSCALER = "Lightricks/ltxv-spatial-upscaler-0.9.7"

def hardware_report():
    assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU."
    cap  = torch.cuda.get_device_capability(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    disk = shutil.disk_usage("/content").free / 1e9
    bf16 = cap[0] >= 8
    print(f"GPU: {torch.cuda.get_device_name(0)} | compute {cap[0]}.{cap[1]} | "
          f"{vram:.1f} GB VRAM | {disk:.0f} GB disk | bf16={'yes' if bf16 else 'NO -> fp16'}")
    if bf16 and vram >= 38 and disk >= 75:
        print("\n>>> This machine CAN run LTX-2.5. Use the full LTX-2.5 script instead.\n")
    return torch.bfloat16 if bf16 else torch.float16

DTYPE = hardware_report()

In [ ]:
from diffusers import DiffusionPipeline, LTXPipeline, LTXImageToVideoPipeline
from diffusers.utils import export_to_video, load_image

def load_first_available(candidates, dtype):
    """Repo names drift between releases; try in order rather than hardcoding one."""
    last = None
    for repo, family in candidates:
        try:
            print(f"Trying {repo} ...")
            kwargs = dict(torch_dtype=dtype)
            if CFG["QUANTIZE_TE"]:
                from transformers import T5EncoderModel, BitsAndBytesConfig
                kwargs["text_encoder"] = T5EncoderModel.from_pretrained(
                    repo, subfolder="text_encoder",
                    quantization_config=BitsAndBytesConfig(load_in_8bit=True),
                    torch_dtype=dtype,
                )
            p = DiffusionPipeline.from_pretrained(repo, **kwargs)
            print(f"Loaded {repo} as {type(p).__name__} ({family})")
            return p, family
        except Exception as e:
            last = e
            print(f"  failed: {type(e).__name__}: {str(e)[:140]}")
    raise RuntimeError(f"No checkpoint loaded. Last error: {last}")

t0 = time.time()
pipe, FAMILY = load_first_available(CANDIDATES, DTYPE)
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()
print(f"Ready in {time.time()-t0:.0f}s")

In [ ]:
def snap_frames(n):  return max(9, ((int(n) - 1) // 8) * 8 + 1)
def snap_dim(x):     return max(32, int(round(x / 32)) * 32)
def seconds_to_frames(s, fps=24): return snap_frames(round(s * fps))

def build_prompt(subject, action, camera, setting, light):
    """LTX wants ONE flowing paragraph, chronological, literal, shot-list style.
    Start with the action. Stay under ~200 words. No audio clause here — 2B is
    video-only; that clause is what you'd add back for LTX-2.5."""
    return (f"{subject} {action} {camera} The scene is {setting}, lit by {light}.")

NEGATIVE = ("worst quality, inconsistent motion, blurry, jittery, distorted, "
            "watermark, text, low resolution")

def free_mem():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

def show_video(path, width=640):
    b64 = base64.b64encode(open(path, "rb").read()).decode()
    display(HTML(f'<video width={width} controls autoplay loop playsinline '
                 f'src="data:video/mp4;base64,{b64}"></video>'))

def sampling_args():
    """Distilled checkpoints are trained for few-step UNGUIDED sampling; passing a
    high step count + CFG to them wastes compute and degrades output, and vice versa."""
    if FAMILY == "distilled":
        return dict(num_inference_steps=8, guidance_scale=1.0)
    return dict(num_inference_steps=40, guidance_scale=3.0, negative_prompt=NEGATIVE)

In [ ]:
def generate(prompt, out="ltx.mp4", image=None, height=None, width=None,
             num_frames=None, seed=None, two_stage=False):
    h  = snap_dim(height or CFG["HEIGHT"])
    w  = snap_dim(width  or CFG["WIDTH"])
    nf = snap_frames(num_frames or CFG["NUM_FRAMES"])
    gen = torch.Generator("cuda").manual_seed(CFG["SEED"] if seed is None else seed)
    free_mem()

    active = pipe
    call = dict(prompt=prompt, generator=gen, **sampling_args())
    call.update(decode_timestep=0.03, decode_noise_scale=0.025)
    if image is not None:
        active = LTXImageToVideoPipeline.from_pipe(pipe)
        call["image"] = load_image(image) if isinstance(image, str) else image
    else:
        active = LTXPipeline.from_pipe(pipe) if not isinstance(pipe, LTXPipeline) else pipe

    tag = "i2v" if image is not None else "t2v"
    print(f"[{tag}] {w}x{h} {nf}f ({nf/CFG['FPS']:.1f}s) "
          f"{'two-stage' if two_stage else 'single-stage'} / {FAMILY}")
    t = time.time()

    if not two_stage:
        frames = active(height=h, width=w, num_frames=nf, **call).frames[0]
    else:
        from diffusers import LTXLatentUpsamplePipeline
        lat = active(height=snap_dim(h // 2), width=snap_dim(w // 2),
                     num_frames=nf, output_type="latent", **call).frames
        ups = LTXLatentUpsamplePipeline.from_pretrained(
            UPSCALER, vae=pipe.vae, torch_dtype=DTYPE)
        ups.to("cuda")
        up = ups(latents=lat, output_type="latent").frames
        del ups; free_mem()
        tail = dict(call); tail["num_inference_steps"] = 4 if FAMILY == "distilled" else 10
        frames = active(latents=up, num_frames=nf, height=h, width=w,
                        denoise_strength=0.4, **tail).frames[0]

    export_to_video(frames, out, fps=CFG["FPS"])
    print(f"-> {out} in {time.time()-t:.0f}s | peak VRAM "
          f"{torch.cuda.max_memory_allocated()/1e9:.1f} GB")
    free_mem()
    return out

In [ ]:
if CFG["RUN_T2V"]:
    p = build_prompt(
        subject="A red fox picks its way along a frozen riverbank at dawn.",
        action="It pauses mid-stride, ears swivelling toward a sound, then breaks into a "
               "light trot through ankle-deep powder snow, leaving a trail of prints.",
        camera="The camera tracks alongside at a low angle with a slight handheld sway, "
               "bare birch trunks sliding past in the foreground.",
        setting="a silent boreal forest under a pale blue pre-sunrise sky",
        light="soft directional dawn light raking across the snow, casting long cold shadows",
    )
    print(textwrap.fill(p, 100), "\n")
    show_video(generate(p, out="01_t2v.mp4"))

if CFG["RUN_I2V"]:
    show_video(generate(
        "The camera slowly dollies backward as the subject shifts its weight and blinks, "
        "dust motes drifting through a shaft of warm afternoon light, the background "
        "gradually falling out of focus.",
        out="02_i2v.mp4", image=IMAGE_URL,
    ))

if CFG["RUN_TWO_STAGE"]:
    show_video(generate(
        "Rain strikes a puddle on dark asphalt at night, individual droplets crowning and "
        "rippling outward while neon reflections shiver and reform on the water surface. "
        "The camera holds steady inches above the puddle. The scene is a wet city street, "
        "lit by magenta and cyan signage bleeding across the water.",
        out="03_two_stage.mp4", two_stage=True, height=384, width=640,
        num_frames=seconds_to_frames(3.0),
    ))

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.1 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.35.1 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
GPU: Tesla T4 | compute 7.5 | 15.6 GB VRAM | 70 GB disk | bf16=NO -> fp16


/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:298: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Trying Lightricks/LTX-Video-0.9.6-distilled ...
  failed: OSError: Lightricks/LTX-Video-0.9.6-distilled is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If 
Trying Lightricks/LTX-Video-0.9.5 ...


config.json:   0%|          | 0.00/786 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/19.9k [00:00<?, ?B/s]

...

model_index.json:   0%|          | 0.00/412 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loaded Lightricks/LTX-Video-0.9.5 as LTXPipeline (dev)
